# GPT 2.0 — WikiText-2 문자 단위 언어 모델

## 개요

이 노트북은 수업 **notebook_06 (Tiny GPT)** 을 발전시킨 **GPT 2.0** 구현입니다.  
데이터셋을 Tiny Shakespeare에서 **WikiText-2** 로 교체하고, 실제 GPT-2 논문의 핵심 기법들을 적용했습니다.

---

### Tiny GPT (notebook_06) vs GPT 2.0 비교

| 항목 | Tiny GPT (notebook_06) | GPT 2.0 (이 노트북) |
|------|------------------------|---------------------|
| 데이터셋 | Tiny Shakespeare (~1.1M chars) | **WikiText-2 (~10.9M chars)** |
| 활성화 함수 | ReLU | **GELU** |
| Weight Tying | ✗ | **✓ (embedding = lm_head)** |
| LR 스케줄링 | 고정 lr | **Cosine Annealing + Warmup** |
| Gradient Clipping | ✗ | **✓ (max_norm=1.0)** |
| 모델 크기 | emb=128, heads=4, layers=4 | **emb=256, heads=8, layers=6** |
| 가중치 초기화 | PyTorch 기본값 | **N(0, 0.02)** |
| 샘플링 | Softmax | **Temperature + Top-k** |

---

### 데이터셋: WikiText-2

- **Merity et al. (2016)** 이 제안한 Wikipedia 기반 언어 모델 벤치마크 데이터셋
- 학습(train): 약 1,091만 자 / 검증(val): 약 114만 자 / 테스트(test): 약 128만 자
- Tiny Shakespeare와 달리 현대 영어, 백과사전 문체, 다양한 주제를 포함

## 1. 설치 및 임포트

In [ ]:
!pip install -q datasets

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import math
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 2. WikiText-2 데이터 로딩

Hugging Face `datasets` 라이브러리를 통해 WikiText-2 원시 텍스트(raw-v1)를 다운로드합니다.

In [ ]:
ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1')

# 빈 줄 제거 후 텍스트 병합
train_text = '\n'.join(line for line in ds['train']['text']      if line.strip())
val_text   = '\n'.join(line for line in ds['validation']['text'] if line.strip())
test_text  = '\n'.join(line for line in ds['test']['text']       if line.strip())

print(f'Train: {len(train_text):>10,} chars')
print(f'Val:   {len(val_text):>10,} chars')
print(f'Test:  {len(test_text):>10,} chars')
print(f'\n--- 학습 데이터 샘플 (200~700번째 문자) ---')
print(train_text[200:700])

## 3. 어휘(Vocabulary) 구성

수업 노트북들과 동일하게 **문자 단위(character-level)** 토크나이저를 사용합니다.

In [ ]:
all_text = train_text + val_text + test_text
chars = sorted(set(all_text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

print(f'Vocabulary size: {vocab_size}')
print(f'Characters (첫 60개): {repr("".join(chars[:60]))}')

# 텍스트를 정수 텐서로 인코딩
train_data = torch.tensor([stoi[c] for c in train_text], dtype=torch.long)
val_data   = torch.tensor([stoi[c] for c in val_text],   dtype=torch.long)

print(f'\nTrain tokens: {len(train_data):,}')
print(f'Val tokens:   {len(val_data):,}')

## 4. Dataset 및 DataLoader

notebook_04부터 사용하는 **GPT-style dataset** 구조를 그대로 사용합니다.
- `x = [t_1, t_2, ..., t_T]`
- `y = [t_2, t_3, ..., t_{T+1}]`  ← 각 위치에서 다음 토큰을 예측

In [ ]:
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx     : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y


BLOCK_SIZE = 128
BATCH_SIZE = 128

train_dataset = NextTokenDataset(train_data, BLOCK_SIZE)
val_dataset   = NextTokenDataset(val_data,   BLOCK_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

xb, yb = next(iter(train_loader))
print(f'xb.shape: {xb.shape}   # (batch, block_size)')
print(f'yb.shape: {yb.shape}   # (batch, block_size)')
print(f'\n샘플 x[0]: {repr("".join(itos[i.item()] for i in xb[0]))}')
print(f'샘플 y[0]: {repr("".join(itos[i.item()] for i in yb[0]))}')

## 5. GPT 2.0 아키텍처

notebook_06의 TinyGPT 구조를 기반으로 아래 개선사항을 추가합니다.

### 5-1. GELU 활성화 함수

실제 GPT-2 논문(Radford et al., 2019)에서 ReLU 대신 **GELU**를 사용합니다.  
GELU는 입력에 Gaussian CDF를 곱한 smooth한 활성화 함수로, 학습 안정성이 높습니다.

$$\text{GELU}(x) = x \cdot \Phi(x) \approx 0.5x\left(1 + \tanh\!\left[\sqrt{\tfrac{2}{\pi}}\left(x + 0.044715x^3\right)\right]\right)$$

### 5-2. Weight Tying

토큰 embedding 행렬과 출력 projection(lm_head) 행렬을 **공유**합니다.  
파라미터 수를 줄이면서 두 행렬이 같은 의미 공간에서 학습되도록 합니다.

### 5-3. 가중치 초기화

GPT-2 원본 구현을 따라 모든 Linear/Embedding 레이어를 $N(0, 0.02)$로 초기화합니다.

In [ ]:
# ── GELU 활성화 함수 ──────────────────────────────────────────────────────────
class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1.0 + torch.tanh(
            math.sqrt(2.0 / math.pi) * (x + 0.044715 * x.pow(3))
        ))


# ── 단일 Attention Head ────────────────────────────────────────────────────────
class Head(nn.Module):
    def __init__(self, emb_dim, head_size, block_size, dropout):
        super().__init__()
        self.key   = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        # Scaled dot-product attention + causal mask
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v


# ── Multi-Head Attention ───────────────────────────────────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout):
        super().__init__()
        head_size = emb_dim // num_heads
        self.heads   = nn.ModuleList([
            Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)
        ])
        self.proj    = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


# ── Feed-Forward Network (GELU 버전) ──────────────────────────────────────────
class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            GELU(),                            # notebook_06: ReLU → GPT 2.0: GELU
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


# ── Transformer Block ──────────────────────────────────────────────────────────
class Block(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout):
        super().__init__()
        self.ln1  = nn.LayerNorm(emb_dim)
        self.sa   = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2  = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # pre-norm + residual
        x = x + self.ffwd(self.ln2(x))
        return x


# ── GPT 2.0 모델 ──────────────────────────────────────────────────────────────
class GPT2(nn.Module):
    """
    GPT 2.0 — notebook_06 TinyGPT 대비 개선사항:
      1. GELU activation (ReLU -> GELU)
      2. Weight tying (token_embedding.weight == lm_head.weight)
      3. N(0, 0.02) 가중치 초기화
      4. 더 큰 모델: emb_dim=256, num_heads=8, num_layers=6
    """
    def __init__(self, vocab_size, block_size,
                 emb_dim=256, num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.block_size = block_size

        self.token_embedding    = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.drop   = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f    = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size, bias=False)

        # Weight Tying: 두 행렬을 동일한 파라미터로 공유
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, x):
        B, T = x.shape
        pos     = torch.arange(T, device=x.device)
        tok     = self.token_embedding(x)
        pos_emb = self.position_embedding(pos).unsqueeze(0)
        h = self.drop(tok + pos_emb)
        h = self.blocks(h)
        h = self.ln_f(h)
        return self.lm_head(h)           # (B, T, vocab_size)

    def num_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# 모델 초기화
model = GPT2(vocab_size, BLOCK_SIZE).to(device)

logits = model(xb.to(device))
init_loss = F.cross_entropy(logits.transpose(1, 2), yb.to(device)).item()

print(f'GPT 2.0 파라미터 수: {model.num_params():,}')
print(f'logits.shape:        {logits.shape}')
print(f'초기 loss:           {init_loss:.4f}  (이론 최솟값: {math.log(vocab_size):.4f})')

## 6. 학습

### 6-1. Cosine Annealing with Warmup

- 처음 `warmup_steps` 동안은 LR을 선형으로 증가
- 이후 cosine 함수로 감소: $\eta_t = \eta_{\max} \cdot \frac{1 + \cos(\pi \cdot t / T)}{2}$

### 6-2. Gradient Clipping

- `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)` 으로 gradient norm을 1.0으로 제한
- 학습 초반 불안정한 gradient 폭발 방지

In [ ]:
def get_lr(step, max_lr=3e-4, warmup_steps=300, total_steps=5000):
    """Linear warmup + cosine annealing LR schedule"""
    if step < warmup_steps:
        return max_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max_lr * 0.5 * (1.0 + math.cos(math.pi * progress))


@torch.no_grad()
def evaluate(model, loader, device, max_batches=80):
    model.eval()
    total, count = 0.0, 0
    for i, (xb, yb) in enumerate(loader):
        if i >= max_batches:
            break
        logits = model(xb.to(device))
        loss   = F.cross_entropy(logits.transpose(1, 2), yb.to(device))
        total += loss.item()
        count += 1
    model.train()
    return total / count

In [ ]:
MAX_STEPS     = 5000
LOG_INTERVAL  = 100

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1
)

train_losses, val_losses, step_log = [], [], []

train_iter = iter(train_loader)
model.train()

for step in range(MAX_STEPS + 1):
    # LR 스케줄링
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g['lr'] = lr

    # 배치 가져오기 (DataLoader 순환)
    try:
        xb, yb = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        xb, yb = next(train_iter)

    xb, yb = xb.to(device), yb.to(device)

    # Forward + Loss
    logits = model(xb)
    loss   = F.cross_entropy(logits.transpose(1, 2), yb)

    # Backward + Gradient Clipping + Optimizer step
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
    optimizer.step()

    if step % LOG_INTERVAL == 0:
        val_loss = evaluate(model, val_loader, device)
        train_losses.append(loss.item())
        val_losses.append(val_loss)
        step_log.append(step)
        print(f'step {step:5d} | train {loss.item():.4f} | val {val_loss:.4f} | lr {lr:.2e}')

print('\n학습 완료!')

## 7. 학습 곡선 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(step_log, train_losses, label='Train Loss', color='steelblue',  linewidth=2)
axes[0].plot(step_log, val_losses,   label='Val Loss',   color='tomato',     linewidth=2)
axes[0].set_xlabel('Step', fontsize=12)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[0].set_title('GPT 2.0 Training on WikiText-2', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# LR schedule 시각화
lr_steps = list(range(MAX_STEPS + 1))
lr_vals  = [get_lr(s) for s in lr_steps]
axes[1].plot(lr_steps, lr_vals, color='darkorchid', linewidth=2)
axes[1].set_xlabel('Step', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('Cosine Annealing with Warmup', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'최종 Train Loss: {train_losses[-1]:.4f}')
print(f'최종 Val Loss:   {val_losses[-1]:.4f}')

## 8. 텍스트 생성

### Temperature + Top-k Sampling

notebook_06의 단순 softmax 샘플링 대신 두 가지 기법을 추가합니다:

- **Temperature** $\tau$: logit을 $\tau$로 나눠 분포를 조절
  - $\tau < 1.0$: 확률 분포가 집중 → 보수적 생성
  - $\tau > 1.0$: 분포가 평탄화 → 창의적 생성
- **Top-k**: 확률 상위 $k$개 토큰만 남기고 나머지를 $-\infty$ 처리

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens=400, temperature=1.0, top_k=40):
    """Temperature + Top-k 샘플링으로 텍스트 생성"""
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        logits   = model(idx_cond)[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        probs    = F.softmax(logits, dim=-1)
        next_tok = torch.multinomial(probs, num_samples=1)
        idx      = torch.cat([idx, next_tok], dim=1)

    model.train()
    return idx


def encode(text, stoi, device):
    return torch.tensor([[stoi.get(c, 0) for c in text]], dtype=torch.long, device=device)

def decode(tensor, itos):
    return ''.join(itos[i.item()] for i in tensor[0])


# 다양한 temperature 값으로 생성 비교
prompt = 'The history of '

for temp in [0.7, 1.0, 1.3]:
    ctx = encode(prompt, stoi, device)
    gen = generate(model, ctx, max_new_tokens=350, temperature=temp, top_k=40)
    print(f'\n{"=" * 60}')
    print(f'Temperature = {temp}  |  Top-k = 40')
    print('=' * 60)
    print(decode(gen, itos))

In [ ]:
# 다양한 시작 문장으로 생성
prompts = [
    'In the field of ',
    'According to the report , ',
    'The researchers found that ',
]

for prompt in prompts:
    ctx = encode(prompt, stoi, device)
    gen = generate(model, ctx, max_new_tokens=250, temperature=0.8, top_k=40)
    print(f'\n[프롬프트: "{prompt}"]')
    print(decode(gen, itos))
    print('-' * 60)

## 9. 모델 저장

In [ ]:
import json

# 모델 가중치 저장
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size':       vocab_size,
    'block_size':       BLOCK_SIZE,
    'stoi':             stoi,
    'itos':             {str(k): v for k, v in itos.items()},
    'final_train_loss': train_losses[-1],
    'final_val_loss':   val_losses[-1],
}, 'gpt2_wikitext2.pt')

print('모델 저장 완료: gpt2_wikitext2.pt')
print(f'최종 Train Loss: {train_losses[-1]:.4f}')
print(f'최종 Val Loss:   {val_losses[-1]:.4f}')

## 10. 요약 및 비교

### 수업 노트북 vs GPT 2.0 학습 결과

| 모델 | 데이터셋 | 최종 Train Loss |
|------|----------|-----------------|
| Bigram (nb01) | names.txt | ~2.52 |
| MLP (nb02) | names.txt | ~2.26 |
| MLP (nb03) | Shakespeare | ~1.86 |
| TinySeq (nb04) | Shakespeare | ~2.46 |
| TinyAttn (nb05) | Shakespeare | ~2.24 |
| **TinyGPT (nb06)** | Shakespeare | **~1.33** |
| **GPT 2.0 (이 노트북)** | **WikiText-2** | **Train 1.3727 / Val 1.3959** |

---

### GPT 2.0에서 도입한 기법 정리

**1. GELU 활성화 함수**  
ReLU는 음수 입력에서 gradient가 0이 되는 "dying ReLU" 문제가 있습니다.  
GELU는 smooth하고 확률론적 해석이 가능한 활성화 함수로 Transformer 계열에서 표준으로 사용됩니다.

**2. Weight Tying**  
Press & Wolf (2017)이 제안한 기법으로, 입력 embedding과 출력 projection의 가중치를 공유합니다.  
동일한 토큰은 입력과 출력에서 같은 벡터 표현을 가지도록 유도합니다.

**3. Cosine Annealing + Linear Warmup**  
학습 초반 LR이 너무 크면 gradient 폭발이 발생합니다. Warmup으로 안정적으로 시작하고,  
cosine 스케줄로 학습 후반부에 세밀한 파라미터 조정을 가능하게 합니다.

**4. Gradient Clipping**  
Gradient norm이 임계값(1.0)을 초과하면 proportionally 축소합니다.  
RNN/Transformer에서 자주 발생하는 gradient 폭발을 방지합니다.

**5. Temperature + Top-k Sampling**  
단순 softmax 샘플링보다 생성 품질을 세밀하게 제어할 수 있습니다.  
`top_k=40`은 GPT-2 논문 원본 데모에서 사용된 설정입니다.